# Modèle de prédiction du risque de diabète

Notebook d'entraînement , équipe Data Science.

Objectif : à partir d'un profil patient (grossesses, glycémie, tension, IMC, etc.), prédire la présence d'un risque de diabète.

**Statut : pipeline entraîné et validé par validation croisée. Rien n'est packagé ni déployé, et aucun suivi de performance n'existe , c'est l'objet de ce TP.**

In [18]:
%load_ext autoreload
%autoreload 2
    
# Amorçage : se placer à la racine du projet (dossier contenant src/)
import os, sys
while not os.path.isdir('src'):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)
sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd())


import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
import joblib
import mlflow

# param proxy
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)

os.environ["NO_PROXY"] = "localhost,127.0.0.1"

for var in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "http_proxy",
    "https_proxy",
    "NO_PROXY",
    "no_proxy",
]:
    print(var, "=", os.environ.get(var))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
cwd = /home/a453784/simplon-briefs/M5B1
HTTP_PROXY = None
HTTPS_PROXY = None
http_proxy = None
https_proxy = None
NO_PROXY = localhost,127.0.0.1
no_proxy = None


In [19]:
df = pd.read_csv("./data/raw/diabetes_train.csv")
df.head()

,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,age,outcome
0,4,110,64,10,0,22.8,0.298,44,0
1,3,96,96,29,102,26.8,0.300,48,0
2,4,133,79,21,35,29.5,0.176,28,0
3,2,102,60,18,0,32.8,0.050,29,0
4,0,144,66,14,178,27.7,0.232,32,0


## 1. Description des données

| Colonne | Description |
|---|---|
| pregnancies | Nombre de grossesses |
| glucose | Glycémie plasmatique (test de tolérance au glucose) |
| blood_pressure | Tension artérielle diastolique (mm Hg) |
| skin_thickness | Épaisseur du pli cutané tricipital (mm) |
| insulin | Insuline sérique à 2h (mu U/ml) |
| bmi | Indice de masse corporelle |
| diabetes_pedigree | Fonction pedigree du diabète (facteur héréditaire) |
| age | Âge du patient |
| outcome | 1 = risque de diabète présent, 0 = absent |

In [20]:
df.describe()

,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,age,outcome
count,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000
mean,2.495714,116.341429,72.375714,20.797143,85.705714,31.598429,0.398056,35.422857,0.344286
std,1.546137,30.113125,11.651606,9.769679,74.643965,6.940337,0.381068,10.398693,0.475475
min,0.000000,50.000000,40.000000,0.000000,0.000000,15.000000,0.050000,18.000000,0.000000
25%,1.000000,95.000000,64.000000,14.000000,16.000000,26.800000,0.119000,28.000000,0.000000
50%,2.000000,115.000000,73.000000,20.000000,77.000000,32.000000,0.268500,34.000000,0.000000
75%,3.000000,138.000000,80.000000,27.000000,135.000000,36.325000,0.562250,42.250000,1.000000
max,9.000000,207.000000,109.000000,56.000000,350.000000,53.700000,2.257000,71.000000,1.000000


In [21]:
df.isna().sum()

pregnancies          0
glucose              0
blood_pressure       0
skin_thickness       0
insulin              0
bmi                  0
diabetes_pedigree    0
age                  0
outcome              0
dtype: int64

In [22]:
df["outcome"].value_counts(normalize=True)

outcome
0    0.655714
1    0.344286
Name: proportion, dtype: float64

## 2. Déséquilibre de classes

~30% de cas positifs seulement. Un modèle entraîné naïvement a tendance à privilégier la classe majoritaire, ce qui est inacceptable ici : en contexte médical, **un faux négatif (rater un patient à risque) coûte bien plus qu'un faux positif**. Deux leviers sont combinés :

- **Sous-échantillonnage aléatoire** (`RandomUnderSampler`) de la classe majoritaire dans le pipeline d'entraînement, pour rééquilibrer les classes ,  appliqué uniquement sur le jeu d'entraînement de chaque fold, jamais sur le jeu de test. On préfère cette approche à une génération d'exemples synthétiques (type SMOTE) : sur un jeu de données cliniques, interpoler entre patients réels peut créer des profils physiologiquement peu plausibles et introduire un biais difficile à auditer. Le sous-échantillonnage ne fabrique aucune donnée : il se contente de réduire la classe majoritaire, au prix d'une partie de l'information (compensé ici par un jeu d'entraînement de taille suffisante).
- **Le rappel comme métrique d'optimisation** de la recherche d'hyperparamètres, plutôt que l'accuracy, qui serait trompeuse sur des classes déséquilibrées.

## 3. Séparation train / test

In [23]:
FEATURES = [
    "pregnancies", "glucose", "blood_pressure", "skin_thickness",
    "insulin", "bmi", "diabetes_pedigree", "age",
]

X = df[FEATURES]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((560, 8), (140, 8))

## 4. Pipeline de prétraitement et de modélisation

Le pipeline enchaîne : imputation des valeurs manquantes (médiane, par robustesse même si ce jeu n'en comporte pas), standardisation, sous-échantillonnage de la classe majoritaire, puis le classifieur. Le tout est encapsulé dans un seul objet scikit-learn/imblearn, pour garantir que le même prétraitement est appliqué à l'entraînement et à l'inférence et que le ré-échantillonnage ne s'applique jamais au moment de la prédiction (comportement natif d'un `imblearn.pipeline.Pipeline`).

In [24]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("undersampler", RandomUnderSampler(random_state=42)),
    ("model", RandomForestClassifier(random_state=42)),
])
pipeline

,steps,"[('imputer', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True


## 5. Recherche d'hyperparamètres

`GridSearchCV` avec validation croisée stratifiée (5 folds), optimisée sur le **rappel**. La grille reste volontairement restreinte (démonstration pédagogique) ; en production on élargirait la recherche ou on passerait à une recherche aléatoire/bayésienne.

In [ ]:
param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 3, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(pipeline, param_grid=param_grid, scoring="recall", cv=cv, n_jobs=-1)


# Parametrage MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(mlflow.get_tracking_uri())

from mlflow import MlflowClient
client = MlflowClient(
    tracking_uri="http://127.0.0.1:5000"
)

print(client.search_experiments())


experiment_name = "Exp_Diabete_prediction_01"
mlflow.set_experiment(experiment_name)


2026/09/02 14:28:43 INFO mlflow.tracking.fluent: Experiment with name 'Exp_Diabete_prediction_01' does not exist. Creating a new experiment.


HTTP_PROXY = None
HTTPS_PROXY = None
http_proxy = None
https_proxy = None
NO_PROXY = localhost,127.0.0.1
no_proxy = None
http://127.0.0.1:5000
[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1788349518062, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1788349518062, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1788352123182, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788352123182, lifecycle_stage='active', name='Exp_Diabete_prediction_01', tags={}, trace_location=None, workspace='default'>

In [17]:

# code remplacé par l'éxcution de la fonction d'entrainement
# search.fit(X_train, y_train)
# search.best_params_, round(search.best_score_, 4)

# exécution de la fonction d'entrainement
from src.train_model import train_model

best_model = train_model(
    search=search,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    model_name="diabetes-risk-model"
)




🔄 Entraînement: diabetes-risk-model


2026/09/02 14:29:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/02 14:29:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Modèle diabetes-risk-model entraîné et logué avec succès dans MLflow.
   - Accuracy: 0.7000
   - recall: 0.7000
   - F1-Score: 0.7038
   - auc: 0.7038


Successfully registered model 'diabetes-risk-model'.
2026/09/02 14:29:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes-risk-model, version 1
Created version '1' of model 'diabetes-risk-model'.
/home/a453784/simplon-briefs/M5B1/src/train_model.py:188: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Run ID : 004f3d4ce5d04aa591d28155981f21da
Model version : 1
Recall : 0.7
🏃 View run eval_diabetes-risk-model at: http://127.0.0.1:5000/#/experiments/1/runs/004f3d4ce5d04aa591d28155981f21da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## 6. Évaluation du meilleur modèle

In [9]:
best_pipeline = search.best_estimator_

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("recall:  ", round(recall_score(y_test, y_pred), 4))
print("f1:      ", round(f1_score(y_test, y_pred), 4))
print("auc:     ", round(roc_auc_score(y_test, y_proba), 4))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

accuracy: 0.7
recall:   0.625
f1:       0.5882
auc:      0.7978
confusion matrix:
 [[68 24]
 [18 30]]
              precision    recall  f1-score   support

           0       0.79      0.74      0.76        92
           1       0.56      0.62      0.59        48

    accuracy                           0.70       140
   macro avg       0.67      0.68      0.68       140
weighted avg       0.71      0.70      0.70       140



In [25]:
importances = pd.Series(
    best_pipeline.named_steps["model"].feature_importances_, index=FEATURES
).sort_values(ascending=False)
importances

glucose              0.397080
bmi                  0.111220
age                  0.102528
diabetes_pedigree    0.089946
blood_pressure       0.080620
skin_thickness       0.078937
pregnancies          0.078920
insulin              0.060749
dtype: float64

Les variables les plus discriminantes restent cohérentes avec la littérature clinique : `glucose`, `bmi`, `age`, `diabetes_pedigree`.

## 7. Sauvegarde du pipeline

Le pipeline complet (prétraitement + modèle) est sauvegardé en `.pkl`, avec la liste des features attendues. **À partir d'ici, ce n'est plus le travail de la data scientist : le suivi en production (MLflow, monitoring, détection de dérive) et l'intégration continue (CI/CD) sont à la charge de l'équipe MLOps (vous donc).**

In [32]:
joblib.dump({"model": best_pipeline, "features": FEATURES}, "./outputs/diabetes_risk_model.pkl")

['./outputs/diabetes_risk_model.pkl']

In [35]:
from src.test_model import test_model

test_model()


0.6825396825396826